# Split before you fit

## Objectives

- Split customers into train, validation, and test without peeking at the test rows.
- Know which set fits the model, which chooses the threshold, and which is scored once.
- Score a majority-class baseline so later models have something to beat.

## What we will do

1. Open `lib/split.py` and read the two stratified splits. The row index checks for overlap between sets. It is not a feature or a customer id.
2. Load the CSV, run `split_customers`, and confirm train / validation / test sizes.
3. Compute majority accuracy on the train rows.
4. Name which set chooses the voucher threshold.
5. Optional: sketch a three-rule paper tree, then continue to `03_train_cv.ipynb`.

Train rows fit the model. Validation rows choose the model and threshold. Test rows are scored once in notebook 05 for churn and once in notebook 06 for spend.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
overlap = len(set(train.index) & set(test.index))
print("overlap train and test", overlap)


### Exercise 1

The printout below is the share of each class on the train rows. `majority_accuracy` is the accuracy you get by predicting the larger class for every train row.


In [ ]:
class_share = y_train.value_counts(normalize=True)
print(class_share)
majority_class = int(class_share.idxmax())
majority_accuracy = None


In [ ]:
if majority_accuracy is None:
    raise NotImplementedError("Finish the majority_accuracy exercise.")
print("majority class", majority_class)
print("majority accuracy", round(float(majority_accuracy), 4))


### Exercise 2

The model later outputs a churn probability. The **voucher threshold** turns that probability into yes (send a voucher) or no (leave the customer alone). You pick the threshold by trying your group's costs on held-out rows.

What the threshold is: a cutoff on the churn probability. At or above it, send a voucher; below it, leave the customer alone.

When you use it: pick the cutoff with your group's costs in the model evaluation part, then apply that same cutoff once on the test rows (and later in the form).

Assign `threshold_set` to `"train"`, `"validation"`, or `"test"`.


In [ ]:
threshold_set = ""


In [ ]:
if threshold_set not in {"train", "validation", "test"}:
    raise NotImplementedError('Set threshold_set to "train", "validation", or "test".')
assert threshold_set == "validation"
print("threshold set", threshold_set)


Why `validation`: `train` already fitted the model, so a threshold chosen there looks too good. `test` stays closed until the model evaluation part, so you cannot tune on it. `validation` is the held-out set for choosing the cutoff before the one test score.

## Optional paper tree, five minutes

On paper, write three if statements a person could use to guess churn. Use plan, days since the last order, and order count. Do not type those rules into the model. At the demo, say whether the model agreed with your first rule.
